# TV Show Recommender

This notebook builds a small TV recommendation workflow using Rotten Tomatoes, Playwright, BeautifulSoup, and OpenAI.

Give it a TV category such as `comedy`, `drama`, `sci fi`, or `horror`. It will:

1. Open the Rotten Tomatoes TV browse page for that category with Playwright.
2. Parse highly rated shows from the rendered page.
3. Visit each selected show detail page for a synopsis and network or streaming provider.
4. Ask an OpenAI model to format the scraped facts as a clean markdown recommendation list.

The model is only used for presentation. The show names, ratings, providers, and synopses come from scraped Rotten Tomatoes page content.

## Environment Setup

This notebook does **not** assume Playwright is already installed.

You have two setup options:

1. Run the one-time install cell below (recommended for notebook users).
2. Or install from terminal at repo root:

```bash
uv add playwright
uv run playwright install chromium
```

Make sure your `.env` file contains an OpenAI API key:

```bash
OPENAI_API_KEY=sk-proj-your-key-here
```

Then select the repo `.venv` kernel and run cells from top to bottom. If you install dependencies mid-session, restart the kernel before rerunning imports.

In [10]:
# One-time setup cell: installs Playwright package + Chromium browser if missing.
# Safe to run multiple times.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("playwright") is None:
    print("Installing playwright package...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "playwright"])
else:
    print("playwright package already installed")

print("Ensuring Chromium browser is installed for Playwright...")
subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
print("Playwright setup complete. If imports fail, restart the kernel and rerun.")

playwright package already installed
Ensuring Chromium browser is installed for Playwright...
Playwright setup complete. If imports fail, restart the kernel and rerun.


In [11]:
import json
import os
import re
from datetime import datetime
from urllib.parse import urljoin

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

try:
    from playwright.async_api import TimeoutError as PlaywrightTimeoutError
    from playwright.async_api import async_playwright
except ImportError as exc:
    raise ImportError(
        "Playwright is not installed in this kernel. Run the setup cell above, restart the kernel, and run this cell again."
    ) from exc

load_dotenv(override=True)
openai = OpenAI()

MODEL = "gpt-4.1-mini"

## How the Workflow Works

Rotten Tomatoes browse pages are JavaScript-heavy, so `requests` alone can miss the show listings. Playwright launches a headless Chromium browser, waits for the page to render, and hands the HTML to BeautifulSoup.

The scraper keeps the facts structured:

- `name`
- `url`
- `critic_score`
- `audience_score`
- `average_rating`
- `provider_or_network`
- `synopsis`

The final OpenAI call turns those facts into a readable recommendation list without inventing missing data.

In [12]:
ROTTEN_TOMATOES_BASE_URL = "https://www.rottentomatoes.com"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

TV_CATEGORY_SLUGS = {
    "action": "action",
    "adventure": "adventure",
    "animation": "animation",
    "anime": "anime",
    "comedy": "comedy",
    "crime": "crime",
    "documentary": "documentary",
    "drama": "drama",
    "fantasy": "fantasy",
    "game show": "game_show",
    "horror": "horror",
    "kids": "kids_family",
    "kids and family": "kids_family",
    "mystery": "mystery_thriller",
    "reality": "reality",
    "romance": "romance",
    "sci fi": "sci_fi",
    "science fiction": "sci_fi",
    "thriller": "mystery_thriller",
}

In [13]:
def tv_category_url(category):
    """Create a Rotten Tomatoes TV browse URL from a friendly category name."""
    normalized = category.strip().lower().replace("-", " ")
    slug = TV_CATEGORY_SLUGS.get(normalized, normalized.replace(" ", "_"))
    return f"{ROTTEN_TOMATOES_BASE_URL}/browse/tv_series_browse/genres:{slug}~sort:popular"


async def fetch_rendered_soup(url, wait_ms=1500, timeout_ms=45_000):
    """Fetch a JavaScript-rendered page with Playwright and return BeautifulSoup."""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(user_agent=headers["User-Agent"], viewport={"width": 1400, "height": 1000})
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
            try:
                await page.wait_for_load_state("networkidle", timeout=10_000)
            except PlaywrightTimeoutError:
                pass
            await page.wait_for_timeout(wait_ms)
            html = await page.content()
        finally:
            await browser.close()
    return BeautifulSoup(html, "html.parser")


def clean_text(value):
    return re.sub(r"\s+", " ", value or "").strip()


def parse_percent(value):
    if value is None:
        return None
    match = re.search(r"(\d{1,3})", str(value))
    if not match:
        return None
    percent = int(match.group(1))
    return percent if 0 <= percent <= 100 else None


def average_percent(*scores):
    available_scores = [score for score in scores if score is not None]
    if not available_scores:
        return None
    return round(sum(available_scores) / len(available_scores))

In [14]:
CURRENT_WINDOW_DAYS = 120


def split_show_title(title):
    """Split season-qualified titles into base name and optional season context."""
    match = re.match(r"^(.*?):\s*Season\s+(\d+)\s*$", title, flags=re.IGNORECASE)
    if match:
        base_name = clean_text(match.group(1))
        season_context = f"Season {match.group(2)}"
        return base_name, season_context
    return title, ""


def parse_latest_episode_date(text):
    """Parse month/day strings like 'Apr 29' from browse cards."""
    match = re.search(r"Latest Episode:\s*([A-Za-z]{3}\s+\d{1,2})", text)
    if not match:
        return None

    month_day = match.group(1)
    now = datetime.now()
    parsed = datetime.strptime(f"{month_day} {now.year}", "%b %d %Y")
    # If date lands far in the future, it likely belongs to last year.
    if (parsed - now).days > 30:
        parsed = parsed.replace(year=now.year - 1)
    return parsed


def parse_show_card(link):
    """Parse one Rotten Tomatoes TV listing link into a structured candidate."""
    href = link.get("href", "")
    if not href.startswith("/tv/"):
        return None

    text = clean_text(link.get_text(" ", strip=True)).replace("Watchlist", "").strip()
    if not text:
        return None

    scores = [parse_percent(score) for score in re.findall(r"\b\d{1,3}%", text)[:2]]
    title = re.sub(r"^(?:\d{1,3}%\s*){0,2}", "", text).strip()
    title = re.split(r"\s+Latest Episode:\s+", title)[0].strip()
    title = re.sub(r"\s+Trailer\s*$", "", title).strip()

    if not title or title.lower() in {"view all", "view more"}:
        return None

    base_name, season_context = split_show_title(title)
    critic_score = scores[0] if scores else None
    audience_score = scores[1] if len(scores) > 1 else None

    latest_episode = parse_latest_episode_date(text)
    has_next_episode = "next ep" in text.lower()
    days_since_latest_episode = (datetime.now() - latest_episode).days if latest_episode else None
    is_currently_airing = has_next_episode or (
        days_since_latest_episode is not None and days_since_latest_episode <= CURRENT_WINDOW_DAYS
    )

    return {
        "name": title,
        "base_name": base_name,
        "season_context": season_context,
        "url": urljoin(ROTTEN_TOMATOES_BASE_URL, href),
        "critic_score": critic_score,
        "audience_score": audience_score,
        "average_rating": average_percent(critic_score, audience_score),
        "latest_episode": latest_episode.strftime("%Y-%m-%d") if latest_episode else "",
        "has_next_episode": has_next_episode,
        "days_since_latest_episode": days_since_latest_episode,
        "is_currently_airing": is_currently_airing,
    }


def parse_show_candidates(soup):
    """Collect unique TV show candidates from a Rotten Tomatoes browse page."""
    shows = []
    seen_urls = set()

    for link in soup.select('a[href^="/tv/"]'):
        show = parse_show_card(link)
        if show and show["url"] not in seen_urls:
            shows.append(show)
            seen_urls.add(show["url"])

    shows.sort(
        key=lambda show: (
            show.get("is_currently_airing", False),
            show["critic_score"] is not None,
            show["critic_score"] or -1,
            show["audience_score"] or -1,
        ),
        reverse=True,
    )
    return shows

In [15]:
def text_between(page_text, start_label, end_labels):
    start = page_text.find(start_label)
    if start == -1:
        return ""

    start += len(start_label)
    end_positions = [page_text.find(label, start) for label in end_labels]
    end_positions = [position for position in end_positions if position != -1]
    end = min(end_positions) if end_positions else len(page_text)
    return page_text[start:end].strip(" :-")


def extract_series_info(page_text, label):
    end_labels = [
        "Creator",
        "Executive Producer",
        "Network",
        "Rating",
        "Genre",
        "Original Language",
        "Release Date",
        "Seasons",
        "Cast & Crew",
    ]
    return clean_text(text_between(page_text, label, [item for item in end_labels if item != label]))


def extract_provider_or_network(soup, page_text):
    network = extract_series_info(page_text, "Network")
    if network:
        return network

    provider_names = []
    for image in soup.select("img[alt]"):
        alt_text = clean_text(image.get("alt"))
        if alt_text and alt_text.lower() not in {"rotten tomatoes", "fandango"}:
            provider_names.append(alt_text)

    for link in soup.select("a[href*='watch']"):
        link_text = clean_text(link.get_text(" ", strip=True))
        if link_text and link_text.lower() not in {"where to watch", "watchlist"}:
            provider_names.append(link_text)

    if provider_names:
        return ", ".join(dict.fromkeys(provider_names[:3]))

    return "Unknown"


def extract_show_details(soup):
    page_text = clean_text(soup.get_text(" ", strip=True))
    synopsis = extract_series_info(page_text, "Synopsis")

    if not synopsis:
        description = soup.select_one('meta[name="description"], meta[property="og:description"]')
        synopsis = clean_text(description.get("content")) if description else ""

    return {
        "provider_or_network": extract_provider_or_network(soup, page_text),
        "synopsis": synopsis or "No synopsis found.",
    }

In [16]:
def recommendation_quality_score(show):
    """Score recommendations to prefer currently airing, richer records."""
    is_currently_airing = show.get("is_currently_airing", False)
    provider_known = show.get("provider_or_network") not in {None, "", "Unknown"}
    synopsis = (show.get("synopsis") or "").lower()
    synopsis_known = synopsis not in {"", "no synopsis found."} and not synopsis.startswith("detail page could not be fetched:")
    is_series_level = not show.get("season_context")
    # Smaller days-since value is better for currently active shows.
    recency_rank = -(show.get("days_since_latest_episode") if show.get("days_since_latest_episode") is not None else 10_000)
    return (
        is_currently_airing,
        provider_known,
        synopsis_known,
        recency_rank,
        is_series_level,
        show.get("critic_score") is not None,
        show.get("critic_score") or -1,
        show.get("audience_score") or -1,
    )


async def find_tv_recommendations(category, max_shows=5):
    """Browse Rotten Tomatoes for highly rated TV shows in a category."""
    category_page_url = tv_category_url(category)
    category_soup = await fetch_rendered_soup(category_page_url)
    candidates = parse_show_candidates(category_soup)

    if not candidates:
        raise ValueError(f"No Rotten Tomatoes shows found for category: {category}")

    # Pull extra candidates so deduplication still leaves enough high-quality series.
    pool_size = max(max_shows * 5, max_shows)
    enriched = []

    for show in candidates[:pool_size]:
        record = dict(show)
        try:
            detail_soup = await fetch_rendered_soup(record["url"], wait_ms=750)
            record.update(extract_show_details(detail_soup))
        except Exception as exc:
            record.update({"provider_or_network": "Unknown", "synopsis": f"Detail page could not be fetched: {exc}"})
        enriched.append(record)

    # Deduplicate season-level and series-level variants by normalized title.
    best_by_title = {}
    for show in enriched:
        key = clean_text(show.get("base_name") or show.get("name", "")).lower()
        if not key:
            continue
        existing = best_by_title.get(key)
        if existing is None or recommendation_quality_score(show) > recommendation_quality_score(existing):
            best_by_title[key] = show

    deduped = list(best_by_title.values())
    deduped.sort(key=recommendation_quality_score, reverse=True)

    recommendations = deduped[:max_shows]
    for show in recommendations:
        show["name"] = show.get("base_name") or show.get("name", "")

    return recommendations

In [17]:
tv_recommendation_system_prompt = """
You are a careful TV recommendation assistant.
Format Rotten Tomatoes facts into a polished markdown recommendation list.
Do not invent providers, networks, ratings, or synopsis details. If a field is Unknown, say Unknown.
Prioritize shows that are currently airing or recently active.
Keep each recommendation concise and useful.
"""


def messages_for_tv_recommendations(category, shows):
    facts = json.dumps(shows, indent=2)
    user_prompt = f"""
The user wants highly rated TV show recommendations for this category: {category}

These shows are already ranked to prioritize currently airing or recently active series.
Here are facts scraped from Rotten Tomatoes. Format them as markdown.
For each show, include:
- name
- optional season context (only if present)
- streaming provider or network
- average rating
- short synopsis
- optional latest episode date (only if present)

Facts:
{facts}
"""
    return [
        {"role": "system", "content": tv_recommendation_system_prompt},
        {"role": "user", "content": user_prompt},
    ]


async def recommend_tv_shows(category, max_shows=5):
    shows = await find_tv_recommendations(category, max_shows=max_shows)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for_tv_recommendations(category, shows),
    )
    return response.choices[0].message.content


async def display_tv_recommendations(category, max_shows=5):
    recommendations = await recommend_tv_shows(category, max_shows=max_shows)
    display(Markdown(recommendations))

## Run the Recommender

Change the category to whatever you want to explore. Good starting points include:

- `comedy`
- `drama`
- `sci fi`
- `horror`
- `documentary`
- `kids and family`

`max_shows=5` keeps the browser work reasonably quick. Increase it if you want a longer list.

In [18]:
await display_tv_recommendations("sci fi", max_shows=5)

Here are some highly rated sci-fi TV shows you might enjoy:

- **Pantheon**  
  - Network: AMC+  
  - Average Rating: 98  
  - Synopsis: Bullied teen Maddie begins receiving messages from a mysterious stranger claiming to be her recently deceased father, whose consciousness has been uploaded to the cloud after an experimental brain scan.  

- **Star Wars: Maul - Shadow Lord**  
  - Network: Disney+  
  - Average Rating: 94  
  - Synopsis: After the Clone Wars, Maul plots to rebuild his criminal syndicate on a planet untouched by the Empire.  

- **Andor**  
  - Network: Disney+  
  - Average Rating: 92  
  - Synopsis: The story of Rebel spy Cassian Andor's formative years of the Rebellion and his difficult missions for the cause.  

- **From**  
  - Network: MGM+  
  - Average Rating: 89  
  - Synopsis: The residents of a small town search for a way out when unknown forces keep them from leaving.  

- **Severance**  
  - Network: Apple TV  
  - Average Rating: 88  
  - Synopsis: Mark leads a team of office workers whose memories have been surgically divided between their work and personal lives; a mysterious colleague appears, triggering a journey to uncover the truth about their jobs.

## Troubleshooting

- `ModuleNotFoundError: No module named 'playwright'`: run `uv add playwright`, restart the notebook kernel, then rerun the imports.
- Browser launch errors: run `uv run playwright install chromium` from the repo root.
- `OPENAI_API_KEY` errors: confirm your `.env` file is in the repo root and contains a valid key.
- Empty or odd Rotten Tomatoes results: try a simpler category like `comedy` or `drama`; Rotten Tomatoes page structure can change over time.
- Slow first run: Playwright has to launch Chromium and visit multiple detail pages. Reduce `max_shows` to `3` while testing.